[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusicalInformatics/miws26/blob/main/expressive_rendering/06_evaluation.ipynb)

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

HOME_DIR = "."

if IN_COLAB:
    import os
    import sys

    # Install system dependency (FluidSynth executable)
    ! apt-get update -qq
    ! apt-get install -y -qq fluidsynth

    # Install python packages
    ! pip install git+https://github.com/CPJKU/partitura.git@develop pyfluidsynth appdirs

    # To be able to access helper modules and data in the repo for this tutorial
    # (not necessary if the jupyter notebook is run locally instead of google colab)
    if not os.path.exists("./miws26"):
        !git clone --recurse-submodules https://github.com/MusicalInformatics/miws26.git
    # RuMEx is a git submodule of the repository (this also gets it if the repository was already cloned)
    !git -C miws26 submodule update --init --recursive
    sys.path.insert(0, "/content/miws26/expressive_rendering/")
    sys.path.insert(0, "/content/miws26/expressive_rendering/rumex/")
    HOME_DIR = "/content/miws26/expressive_rendering/"

# Evaluating expressive performances

How different are the performances of two pianists? And how close is a generated performance to a real one? To answer these questions we compare the **expressive parameters** of the performances, computed with the performance codec that we saw in the first notebook.

1. We compare the performances of the same piece in the Vienna 4x22 dataset.
2. We look at different definitions of the same expressive parameter (the tempo).
3. We evaluate two systems that generate performances, the **Basis Mixer** and **RuMEx**, and compare them with the difference between two pianists.

In [ ]:
import os
import sys
import warnings

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from helper import init_dataset
import helper.data as data
from helper import evaluation as ev

%matplotlib inline

warnings.filterwarnings("ignore")
init_dataset()

## 1. Comparing performances of the same piece

The alignment of a performance with its score tells us which performed note corresponds to which note of the score. With it, the performance codec describes each performed note with four expressive parameters:

| parameter | description |
| --- | --- |
| `velocity` | MIDI velocity (loudness) of the note, divided by 127 |
| `beat_period` | duration of a beat in seconds at the onset of the note (the larger, the slower) |
| `timing` | deviation (in seconds) of the note from the average onset time of the notes that start at the same score position; positive values are played earlier |
| `articulation_log` | logarithm (base 2) of the performed duration divided by the duration in the score in the current tempo; negative values are shorter (more *staccato*) |

Since all performances of a piece share the score, we can compare them note by note. In the plots we show one value per score onset: the mean `velocity` and `articulation_log` of its notes, its `beat_period`, and the `timing` of its highest note (the mean `timing` of an onset is zero by definition). Grace notes are left out. The duration of the real performances is the time the keys are held, without the sustain pedal.

Choose a piece and two pianists to highlight. All 22 pianists are shown in gray, and their average in black. The matrices show the correlation of each parameter between every pair of pianists.

In [ ]:
ev.performance_explorer()

Which parameters do the pianists agree on the most? Is it the same in all pieces?

The following figure summarizes all pieces: for each of them, it shows how the average value of a parameter is distributed over the 22 pianists (for the timing, whose average is close to zero, we show how much the top note deviates instead).

In [ ]:
ev.plot_all_pieces();

## 2. Different definitions of the tempo

The same information can be described in different ways. The codec of partitura has several definitions of the tempo. With $b_i$ the beat period at the $i$-th score onset, and $\mu$ and $\sigma$ the mean and standard deviation of the beat periods of the whole performance:

| definition | value | additional parameters (one value per performance) |
| --- | --- | --- |
| `beat_period` | $b_i$ | |
| `beat_period_log` | $\log_2 b_i$ | |
| `beat_period_ratio` | $b_i / \mu$ | `beat_period_mean` ($\mu$) |
| `beat_period_ratio_log` | $\log_2 (b_i / \mu)$ | `beat_period_mean` |
| `beat_period_standardized` | $(b_i - \mu) / \sigma$ | `beat_period_mean`, `beat_period_std` ($\sigma$) |

None of them loses information: with the additional parameters we can always get the beat period back. For example, for the standardized beat period:

In [ ]:
piece = data.PIECES[0]
raw = ev.encoded_performance(piece, "p01", "beat_period")
standardized = ev.encoded_performance(piece, "p01", "beat_period_standardized")

error = max(
    abs(p["beat_period_standardized"] * p["beat_period_std"] + p["beat_period_mean"] - raw[i]["beat_period"])
    for i, p in standardized.items()
)
print(f"largest difference between the beat periods: {error:.1e} s")

But they separate the information in different ways, and this changes what we compare. Select one or more definitions and pianists (ctrl/cmd + click to select several). The left panels show the tempo curves, and the right panels the additional parameters of each pianist. Below the figure, you see how different the curves are on average over **all** pairs of pianists (mean squared error, MSE, and correlation).

In [ ]:
ev.tempo_explorer()

Some questions to think about:

* Which definitions remove the differences in the overall tempo of the pianists? And in how much they vary the tempo?
* Why can two definitions have very different MSEs, but almost the same correlation?
* Which definition would you predict with a model? What do you then need to turn the predictions into a performance?

## 3. Evaluating generated performances

We now compare three things on a **test set**: the performances of pianists `p21` and `p22` of every piece.

* **Basis Mixer**: the model that you trained in the previous notebooks (notebook 4). If you have not trained a model, we use our model in `sample_data/models` instead.
* **RuMEx**: the rule-based system of notebook 3, with its default rules. Some rules are random, so we average the results of three random seeds.
* **Another pianist**: we compare the two test performances with each other. This shows how different two human performances of the same piece are.

For every parameter we compute the **mean squared error (MSE)** (lower is better) and the **Pearson correlation** $r$ (higher is better) between a generated performance and a test performance, over the notes of the score (without grace notes). We first average over the test performances of each piece, and then over the pieces. We use the codec with `beat_period`, i.e., the tempo in seconds per beat.

Keep in mind:

* The MSE depends on the **level** of a parameter (e.g., the overall tempo or loudness), the correlation only on its **shape**.
* The model of notebook 4 predicts `beat_period_standardized`, so it needs a mean and a standard deviation of the beat period to produce a tempo. We take their average over the **other** pianists (`p01`-`p20`) of the same piece. This is information that RuMEx does not get: it chooses its tempo from the tempo marking of the score. (It changes the MSE of `beat_period`, not its correlation.)
* Notebook 4 splits the performances into training and test sets **randomly**, so your model may have seen some of the performances of `p21` and `p22` during training.
* The model of notebook 4 does not predict `velocity_dev` (the loudness differences between the notes of an onset), and the `timing` it learns is the average of the notes of an onset, which is zero by definition. Do not expect much from its `timing`.

In [ ]:
from helper.rendering import (
    load_model,
    compute_basis_from_xml,
    post_process_predictions,
    path_to_trained_models,
)
from basismixer.performance_codec import get_performance_codec

# RuMEx lives in the folder `rumex` (a package with the same name inside it). It is a git submodule:
# if the folder is empty, run `git submodule update --init --recursive` in the repository
sys.path.insert(0, os.path.join(HOME_DIR, "rumex"))
from rumex import load_score, render_piece

TEST_PERFORMERS = ["p21", "p22"]
TRAIN_PERFORMERS = [p for p in data.PERFORMERS if p not in TEST_PERFORMERS]
RUMEX_SEEDS = [0, 1, 2]


def score_file(piece):
    return os.path.join(data.DATASET_DIR, "musicxml", f"{piece}.musicxml")

### 3.1 The Basis Mixer

We load the model of notebook 4. To use our model instead, set `models_dir = os.path.join(HOME_DIR, "sample_data", "models")` (it predicts the mean and standard deviation of the beat period itself).

In [ ]:
models_dir = path_to_trained_models()  # your model of notebook 4 (or ours if there is none)
print(models_dir)
model, predicted_parameters = load_model(models_dir)

In [ ]:
def basis_mixer(piece):
    """Performance of the Basis Mixer, encoded with the performance codec"""
    # `unfold=False`: keep the note ids of the score, which the performances of the dataset use
    basis, part = compute_basis_from_xml(
        score_file(piece), model.input_names, double_notes="keep", unfold=False
    )
    # tempo of the other pianists (only used if the model does not predict these parameters)
    tempo_values = {
        name: np.mean([
            next(iter(ev.encoded_performance(piece, p, "beat_period_standardized").values()))[name]
            for p in TRAIN_PERFORMERS
        ])
        for name in ["beat_period_mean", "beat_period_std"]
    }
    predictions = model.predict(basis, part.note_array()["onset_beat"], default_values=tempo_values)
    post_process_predictions(predictions)
    codec = get_performance_codec(model.output_names)
    ppart, alignment = codec.decode(part, predictions, return_alignment=True)
    return [ev.encode(piece, ppart, alignment)]

### 3.2 RuMEx

In [ ]:
def rumex(piece):
    """Performances of RuMEx (one per seed), encoded with the performance codec"""
    performances = []
    for seed in RUMEX_SEEDS:
        ppart, _, _, _, alignment = render_piece(
            load_score(score_file(piece)), rng=np.random.RandomState(seed), return_features=True
        )
        performances.append(ev.encode(piece, ppart, alignment))
    return performances

### 3.3 Results

In [ ]:
results = ev.evaluate_systems(
    {"Basis Mixer": basis_mixer, "RuMEx": rumex}, test_performers=TEST_PERFORMERS
)

The table has the MSE and the correlation of each parameter, averaged over the pieces. If one of the two performances has a constant value of a parameter, the correlation is not defined; we report 0.

In [ ]:
table = ev.results_table(results)
table.style.format("{:.4f}", subset=[c for c in table.columns if c[1] == "MSE"]).format(
    "{:.2f}", subset=[c for c in table.columns if c[1] == "r"]
)

And the results for each piece:

In [ ]:
by_piece = ev.results_table(results, by_piece=True)
by_piece.style.format("{:.4f}", subset=[c for c in by_piece.columns if c[1] == "MSE"]).format(
    "{:.2f}", subset=[c for c in by_piece.columns if c[1] == "r"]
)

Some questions to think about:

* For which parameters are the models closer to a test pianist than the other pianist is? Does this mean that they play better than a human?
* Compare the MSE and the correlation of RuMEx. What do they tell you about its rules?
* What would you need to say which system sounds better?

## 4. Good numbers, bad performance

Can we trust these metrics? Let us build a performance in which some expressive parameters are **random**. We split the piece into 4 segments of equal length. In each segment, we take the mean and the standard deviation of a parameter of every pianist, and average them over the pianists: the mean $\mu$ and the standard deviation $\sigma$ of a typical pianist in that segment. The values of the parameter in the segment are then drawn from a **uniform distribution**,

$$y \sim \mathcal{U}(\mu - \alpha\,\sigma,\; \mu + \alpha\,\sigma),$$

one value per onset for the tempo and one per note for the other parameters. Within a segment, the values have nothing to do with the notes: there are no phrases, no accents, and no relation between neighboring notes. No pianist plays like this. All the parameters that we do not make random stay as in the average performance.

With the weight $\alpha = 0.8$, for the Schubert piece, a random **beat period** or **timing** is still **closer to the average performance** (smaller MSE) than some of the real pianists. This does not work for the **velocity** and the **articulation**, whose random versions are farther from the average than all pianists. In the other pieces, it depends on the parameter.

**A constant prediction.** As a second extreme case, we "predict" a **constant** value for the parameter: its mean in the average performance. This performance has no expression at all (e.g., a constant tempo is a metronome). Its correlation with any performance is not defined, since a constant does not vary (we report 0). But what about its MSE?

(Note that we need statistics of real performances of the piece to do this: this is not a system that could perform a new piece. It is a test for our metrics.)

### 4.1 Make a parameter random

Choose **which parameters are random**. Start with one at a time, to hear what each of them does. For each random parameter, the left panel shows its values (the average performance, the random one, the constant one and a real pianist) in the part of the piece that you hear. The dotted lines are the limits of the segments, and the red areas the ranges from which the random values are drawn. (As in section 1, the curves show one value per onset.) The right panel shows the MSE to the average of every pianist and of the random and constant performances, computed on the whole piece. **How many pianists are farther from the average than the random performance? And than the constant one?**

Then listen to the four versions of the first 30 seconds (of the average performance), all synthesized in the same way with FluidSynth (without sustain pedal):

1. the average performance,
2. the average performance with your random parameters,
3. the average performance with constant values for the same parameters, and
4. a real pianist. With *automatic*, we pick the one who is farther from the average than the random performance in as many of the chosen parameters as possible (if nobody is farther, the one who is farthest compared with it). Where this pianist is farther, the MSE says that version 2 is "better".

*Alpha* sets the weight $\alpha$, i.e., how wide the ranges of the random values are (with $0$, each segment has a constant value, its mean $\mu$). *New random values* draws a different random performance.

In [ ]:
ev.random_explorer()

Some questions to think about:

* Which random parameter sounds the most unnatural? Is it the one for which the MSE is the worst, compared with the pianists?
* Move the *Alpha* slider. What happens to the numbers and the sound? For which values of $\alpha$ does the random performance still beat some pianists?
* Why do a random velocity and a random articulation get a much worse MSE, compared with the pianists, than a random tempo or timing?
* Try the other pieces. Where does the random performance beat many pianists according to the MSE?
* The constant performance has no expression. How does its MSE compare with the pianists and with the random performance? What does its correlation tell you?
* Look at the correlation in the table. Does it tell the random performance and the pianist apart better than the MSE?

### 4.2 The random performance on the test set

Finally, we add the random and the constant performances to the comparison of section 3. To be fair, its means and standard deviations are computed only with the pianists of the training set (`p01`-`p20`). By default all four parameters are random; the ones that are not in `RANDOM_PARAMETERS` are the ones of the average.

In [ ]:
RANDOM_PARAMETERS = ev.PARAMETERS  # e.g. ["beat_period"] to make only the tempo random


def random_performance(piece):
    """Random performances (one per seed), with the means and stds of the training pianists"""
    return [
        ev.random_parameters(
            piece, seed=seed, performers=TRAIN_PERFORMERS,
            parameters=RANDOM_PARAMETERS,
        )
        for seed in RUMEX_SEEDS
    ]


def constant_performance(piece):
    """Constant values (the means of the training pianists) for the same parameters"""
    return [ev.constant_parameters(piece, performers=TRAIN_PERFORMERS, parameters=RANDOM_PARAMETERS)]


results_with_random = ev.evaluate_systems(
    {
        "Basis Mixer": basis_mixer,
        "RuMEx": rumex,
        "Random": random_performance,
        "Constant": constant_performance,
    },
    test_performers=TEST_PERFORMERS,
)
table = ev.results_table(results_with_random)
table.style.format("{:.4f}", subset=[c for c in table.columns if c[1] == "MSE"]).format(
    "{:.2f}", subset=[c for c in table.columns if c[1] == "r"]
)

Compare what you heard with the numbers:

* Which rows of the table are "better" than the random performance, and in which parameters? Does this agree with what you hear?
* Which systems have a larger MSE than the constant performance? Would you say that they are worse than playing without any expression?
* The MSE treats every note on its own. What does it miss? Does the correlation do better?
* Change `RANDOM_PARAMETERS`. Which parameter changes the numbers the most? Is it the same one that changes the sound the most?
* What else could we measure (or do) to evaluate expressive performances?